# Three-Way Architecture Analysis

Aggregates training results from `runs/*/` (one subdir per training run,
containing `config.json` + `metrics.csv` written by `src.train`) into:

1. **Comparison table** — config × {best mIoU, params (M), best-epoch,
   train time, per-class IoU}.
2. **Pareto plot** — params vs best mIoU, one point per run.
3. **Per-class IoU bar chart** — grouped by config, helpful for spotting
   which architecture wins on the under-represented classes
   (`living_obs`, `bridge`).
4. **Convergence curves** — val mIoU over epochs, all runs on one plot.

Designed to be runnable on a laptop after copying the `runs/` artifacts
down from Kaggle. No GPU needed.

All figures are saved under `runs/analysis/` as both PNG (for slides) and
PDF (for the thesis).

## 1. Config — point this at your runs/ directory

Default is `<repo_root>/runs`. Override `RUNS_GLOB` to filter by name
(e.g. only the three thesis configs).

In [ ]:
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
RUNS_DIR = REPO_ROOT / 'runs'
ANALYSIS_DIR = RUNS_DIR / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# Glob for run directories. Each must contain config.json + metrics.csv.
# Adjust to filter by name, e.g. 'unet_vgg16_*' or 'custom_lwir_*'.
RUNS_GLOB = '*/'

print('Repo root:    ', REPO_ROOT)
print('Runs dir:     ', RUNS_DIR)
print('Analysis dir: ', ANALYSIS_DIR)
print('Glob pattern: ', RUNS_GLOB)

## 2. Load all runs

Each run directory is expected to contain `config.json` (resolved
`TrainConfig`) and `metrics.csv` (one row per epoch). Runs missing
either file are skipped with a warning.

In [ ]:
import json
import pandas as pd

def label_for(cfg: dict) -> str:
    """Human-readable config label from a TrainConfig dict."""
    model = cfg.get('model', 'unknown')
    pretrained = cfg.get('encoder_weights') is not None
    if model == 'unet_vgg16':
        return 'vgg16_pretrained' if pretrained else 'vgg16_scratch'
    if model == 'custom_lwir':
        return 'custom_lwir'
    if model == 'vgg16_ext':
        parts = ['vgg16_ext']
        if cfg.get('use_transformer_bottleneck'): parts.append('trans')
        if cfg.get('use_attention_gates'):        parts.append('att')
        if cfg.get('use_aux_heads'):              parts.append('aux')
        return '+'.join(parts)
    return model

runs = []
for run_dir in sorted(RUNS_DIR.glob(RUNS_GLOB)):
    if not run_dir.is_dir() or run_dir.name == 'analysis':
        continue
    cfg_path = run_dir / 'config.json'
    metrics_path = run_dir / 'metrics.csv'
    if not cfg_path.exists() or not metrics_path.exists():
        print(f'skip {run_dir.name}: missing config.json or metrics.csv')
        continue
    cfg = json.loads(cfg_path.read_text())
    df = pd.read_csv(metrics_path)
    runs.append({
        'run_dir': run_dir,
        'name':    run_dir.name,
        'label':   label_for(cfg),
        'config':  cfg,
        'metrics': df,
    })

print(f'Loaded {len(runs)} runs:')
for r in runs:
    print(f'  {r["label"]:20s}  ({r["name"]})  epochs={len(r["metrics"])}')

## 3. Comparison table

One row per run. Best-epoch metrics, parameter count, total train time.

In [ ]:
# Build the model once per config to read its parameter count. Cheap on CPU.
# Imported here (not at top) so the data-loading cells run even on a machine
# without torch installed.
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.train import TrainConfig, build_model

def param_count_for(cfg_dict: dict) -> float:
    """Total parameter count (millions) for the resolved TrainConfig dict."""
    # Drop fields that aren't constructor args of the current TrainConfig
    # (older runs may have written extra/legacy keys).
    allowed = set(TrainConfig.__dataclass_fields__.keys())
    args = {k: v for k, v in cfg_dict.items() if k in allowed}
    cfg = TrainConfig(**args)
    model = build_model(cfg)
    n = sum(p.numel() for p in model.parameters())
    del model
    return n / 1e6

In [ ]:
CLASS_NAMES = ['sky', 'water', 'bridge', 'obstacle', 'living_obs', 'background', 'self']

rows = []
for r in runs:
    df = r['metrics']
    if df.empty:
        continue
    best = df.loc[df['mIoU'].idxmax()]
    try:
        params_m = param_count_for(r['config'])
    except Exception as e:
        print(f'  {r["label"]}: param count unavailable ({e})')
        params_m = float('nan')
    row = {
        'config':       r['label'],
        'name':         r['name'],
        'params_M':     round(params_m, 2),
        'best_mIoU':    round(float(best['mIoU']), 4),
        'best_epoch':   int(best['epoch']),
        'train_time_min': round(df['elapsed_s'].sum() / 60.0, 1),
    }
    for c in CLASS_NAMES:
        col = f'iou_{c}'
        if col in best.index and best[col] == best[col]:
            row[c] = round(float(best[col]), 4)
    rows.append(row)

table = pd.DataFrame(rows).sort_values('best_mIoU', ascending=False)
print(table.to_string(index=False))

# CSV mirror for the thesis appendix.
table.to_csv(ANALYSIS_DIR / 'comparison_table.csv', index=False)
print(f'\nSaved: {ANALYSIS_DIR / "comparison_table.csv"}')

## 4. Pareto plot — params (M) vs best mIoU

Points up-and-to-the-left dominate (more accurate, fewer parameters).
The custom_lwir architecture's goal is to sit clearly in that quadrant.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
for _, row in table.iterrows():
    ax.scatter(row['params_M'], row['best_mIoU'], s=80)
    ax.annotate(row['config'], (row['params_M'], row['best_mIoU']),
                xytext=(6, 4), textcoords='offset points', fontsize=9)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Best val mIoU')
ax.set_title('Pareto: efficiency vs accuracy')
ax.grid(alpha=0.3)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'pareto.{ext}', dpi=150)
plt.show()

## 5. Per-class IoU bar chart

Grouped by class, one bar per config. `living_obs` (~0.05% of pixels in
MassMIND) is where the architectures usually diverge most.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

configs = table['config'].tolist()
classes = [c for c in CLASS_NAMES if c in table.columns]

x = np.arange(len(classes))
w = 0.8 / max(len(configs), 1)

fig, ax = plt.subplots(figsize=(10, 5))
for i, cfg_name in enumerate(configs):
    vals = table.loc[table['config'] == cfg_name, classes].values.flatten()
    ax.bar(x + i * w - 0.4 + w / 2, vals, w, label=cfg_name)
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=20, ha='right')
ax.set_ylabel('IoU at best epoch')
ax.set_title('Per-class IoU by config')
ax.set_ylim(0, 1)
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'per_class_iou.{ext}', dpi=150)
plt.show()

## 6. Convergence curves

Val mIoU over epochs, all runs overlayed. Reveals which configs converge
fastest, plateau early, or are still climbing at end of training.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for r in runs:
    df = r['metrics']
    if df.empty: continue
    ax.plot(df['epoch'], df['mIoU'], marker='o', markersize=3, label=r['label'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Val mIoU')
ax.set_title('Convergence — val mIoU per epoch')
ax.grid(alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(ANALYSIS_DIR / f'convergence.{ext}', dpi=150)
plt.show()

## 7. Done

Artifacts saved under `runs/analysis/`:

- `comparison_table.csv` — the table from §3
- `pareto.{png,pdf}` — §4
- `per_class_iou.{png,pdf}` — §5
- `convergence.{png,pdf}` — §6

Drop these straight into the thesis. The CSV reads cleanly into LaTeX via
`pgfplotstable` or `siunitx`.